In [ ]:
import pickle
import numpy as np
import tensorflow as tf
import keras
from cnn_base import build_base_model, build_qkeras_model, build_t1, build_t1_quantized
import os


TRAIN_PICKLE_PATH = "./dataset/sat6_train.pkl"
TEST_PICKLE_PATH = "./dataset/sat6_test.pkl"
DATA_DIR = "./dataset/sat-6-full.mat"
BATCH_SIZE = 512
EPOCHS = 30
LEARNING_RATE = 3e-4
 
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

def trace_layers(config):
    config['Model']['TraceOutput'] = True
    for layer in config['LayerName']:
        config['LayerName'][layer]['Trace'] = True


def print_dict(d, indent=0):
    for key, value in d.items():
        print('  ' * indent + str(key), end='')
        if isinstance(value, dict):
            print()
            print_dict(value, indent + 1)
        else:
            print(':' + ' ' * (20 - len(key) - 2 * indent) + str(value))
 
def load_pickle_data(path):
    print(f"Lendo {path} ...")
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"], data["y"]

In [2]:
X_train_all, y_train_all = load_pickle_data(TRAIN_PICKLE_PATH)
X_test, y_test = load_pickle_data(TEST_PICKLE_PATH)

X_train_all = X_train_all.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

val_idx = int(0.75 * len(X_train_all))
X_train, y_train = X_train_all[:val_idx], y_train_all[:val_idx]
X_val, y_val = X_train_all[val_idx:], y_train_all[val_idx:]

n_classes = 6
input_shape = X_train.shape[1:]
train_size = len(X_train)
input_shape, train_size

Lendo ./dataset/sat6_train.pkl ...
Lendo ./dataset/sat6_test.pkl ...


((32, 32, 4), 243000)

# Train T1 base model

In [ ]:
# Finn base model not quantized
model = build_t1()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy")
    ],
)
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
]

model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks)


os.makedirs('models', exist_ok=True)
os.makedirs('models/t1', exist_ok=True)
base_path = './models/t1/'
model.save(base_path + '/model.keras')
model.save(base_path + '/model.h5')
model.save_weights(base_path + '/weights.h5')

with open(base_path + '/model.json', 'w') as f:
    f.write(model.to_json())

In [3]:
# Finn base model not quantized with batch normalization and max pooling
model = build_t1(batch_norm=True, max_pool=True)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy")
    ],
)
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
]

model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks)


os.makedirs('models', exist_ok=True)
os.makedirs('models/t1_batch_norm_max_pool', exist_ok=True)
base_path = './models/t1_batch_norm_max_pool'
model.save(base_path + '/model.keras')
model.save(base_path + '/model.h5')
model.save_weights(base_path + '/weights.h5')

with open(base_path + '/model.json', 'w') as f:
    f.write(model.to_json())

Epoch 1/30
7594/7594 [==============================] - 12s 1ms/step - loss: 0.1399 - accuracy: 0.9518 - val_loss: 0.0762 - val_accuracy: 0.9732 - lr: 3.0000e-04
Epoch 2/30
7594/7594 [==============================] - 11s 1ms/step - loss: 0.0692 - accuracy: 0.9755 - val_loss: 0.0552 - val_accuracy: 0.9816 - lr: 3.0000e-04
Epoch 3/30
7594/7594 [==============================] - 11s 1ms/step - loss: 0.0585 - accuracy: 0.9791 - val_loss: 0.0499 - val_accuracy: 0.9831 - lr: 3.0000e-04
Epoch 4/30
7594/7594 [==============================] - 11s 1ms/step - loss: 0.0529 - accuracy: 0.9815 - val_loss: 0.0557 - val_accuracy: 0.9804 - lr: 3.0000e-04
Epoch 5/30
7594/7594 [==============================] - 11s 1ms/step - loss: 0.0486 - accuracy: 0.9830 - val_loss: 0.0544 - val_accuracy: 0.9812 - lr: 3.0000e-04
Epoch 6/30
7594/7594 [==============================] - 11s 1ms/step - loss: 0.0451 - accuracy: 0.9846 - val_loss: 0.1676 - val_accuracy: 0.9479 - lr: 3.0000e-04
Epoch 7/30
7594/7594 [======

/home/fauri/Documentos/hls4ml/.venv_hls4ml_t/lib/python3.10/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:
# My Base model not quantized
model = build_base_model()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy")
    ],
)
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
]

model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks)


os.makedirs('models', exist_ok=True)
os.makedirs('models/base', exist_ok=True)
base_path = './models/base/'
model.save(base_path + '/model.keras')
model.save(base_path + '/model.h5')
model.save_weights(base_path + '/weights.h5')

with open(base_path + '/model.json', 'w') as f:
    f.write(model.to_json())

## Tensorflow quantization

In [ ]:
import tensorflow_model_optimization as tfmot

quantize_model = tfmot.quantization.keras.quantize_model
model = build_base_model()
q_aware_model = quantize_model(model)

q_aware_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy")
    ],
)
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
]

q_aware_model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks)


In [ ]:
stripped_model = tfmot.quantization.keras.strip_quantization(
    q_aware_model
)

converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

quantized_tflite_model = converter.convert()

base_path = "./models/base_model_qa"
os.makedirs(base_path, exist_ok=True)

with open(f"{base_path}/model.tflite", "wb") as f:
    f.write(quantized_tflite_model)

q_aware_model.save(f"{base_path}/model.h5")
stripped_model.save(f"{base_path}/stripped_model.h5")
q_aware_model.save(f"{base_path}/model.keras")
q_aware_model.save_weights(f"{base_path}/weights.h5")
with open(f"{base_path}/model.json", "w") as f:
    f.write(q_aware_model.to_json())


# Finn Model

In [ ]:
# t1 do paper FINN (cnn_base.build_t1_quantized): conv 3x3/2 -> conv 1x1 -> dense, sem bias
bits = [8, 4]

for bit in bits:
    model = build_t1_quantized(input_shape=input_shape, n_classes=n_classes, bits=bit)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name="accuracy")
        ],
    )
    callbacks = [
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    ]

    model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks)


    os.makedirs('models', exist_ok=True)
    model_name = f't1_{bit}'
    os.makedirs(f'models/{model_name}', exist_ok=True)
    base_path = f'./models/{model_name}/'
    model.save(base_path + '/model.keras')
    model.save(base_path + '/model.h5')
    model.save_weights(base_path + '/weights.h5')

    with open(base_path + '/model.json', 'w') as f:
        f.write(model.to_json())

In [ ]:
# mesma t1 quantizada com max pooling e batch normalization depois de cada conv
bits = [8, 4]

for bit in bits:
    model = build_t1_quantized(
        input_shape=input_shape,
        n_classes=n_classes,
        bits=bit,
        batch_norm=True,
        max_pool=True,
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name="accuracy")
        ],
    )
    callbacks = [
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    ]

    model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks)


    os.makedirs('models', exist_ok=True)
    model_name = f't1_{bit}_mp_bn'
    os.makedirs(f'models/{model_name}', exist_ok=True)
    base_path = f'./models/{model_name}/'
    model.save(base_path + '/model.keras')
    model.save(base_path + '/model.h5')
    model.save_weights(base_path + '/weights.h5')

    with open(base_path + '/model.json', 'w') as f:
        f.write(model.to_json())

# My Model

In [ ]:
filters_per_conv_layer = [16, 32, 32]
neurons_per_dense_layer = [32]
bits = [8,4]

for bit in bits:
    model = build_qkeras_model(
        input_shape=input_shape,
        n_classes=n_classes,
        filters_per_conv_layer=filters_per_conv_layer,
        neurons_per_dense_layer=neurons_per_dense_layer,
        bits=bit,
        kernel_size=(3,3),
        pool_size=(2, 2),
        max_pool=True,
        batch_norm=True,
        use_bias=False,
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name="accuracy")
        ],
    )
    callbacks = [
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    ]

    model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks)


    os.makedirs('models', exist_ok=True)
    model_name = f'base_{bit}'
    os.makedirs(f'models/{model_name}', exist_ok=True)
    base_path = f'./models/{model_name}/'
    model.save(base_path + '/model.keras')
    model.save(base_path + '/model.h5')
    model.save_weights(base_path + '/weights.h5')

    with open(base_path + '/model.json', 'w') as f:
        f.write(model.to_json())

# Paper Model

In [ ]:
filters_per_conv_layer = [16, 16, 24]
bits = [8,7,4]
neurons_per_dense_layer = [42,64]

for bit in bits:
    model = build_qkeras_model(
        input_shape=input_shape,
        n_classes=n_classes,
        filters_per_conv_layer=filters_per_conv_layer,
        neurons_per_dense_layer=neurons_per_dense_layer,
        bits=bit,
        kernel_size=(3,3),
        pool_size=(2, 2),
        max_pool=True,
        batch_norm=True,
        use_bias=False,
        #use_integer=True,
        #use_integer_signed=True
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.CategoricalCrossentropy(),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name="accuracy")
        ],
    )
    callbacks = [
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    ]

    model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks)


    os.makedirs('models', exist_ok=True)
    model_name = f'paper_{bit}'
    os.makedirs(f'models/{model_name}', exist_ok=True)
    base_path = f'./models/{model_name}/'
    model.save(base_path + '/model.keras')
    model.save(base_path + '/model.h5')
    model.save_weights(base_path + '/weights.h5')

    with open(base_path + '/model.json', 'w') as f:
        f.write(model.to_json())